# Stage B — Grounded LLM Filter with Dossier Evidence

**Goal**: take Stage A v3's top-2000 per query, screen each candidate with
Qwen3-32B-AWQ + a pre-digested dossier, output keep/drop + verbatim evidence
quote. The quote-substring check is a hard gate against hallucinated picks.

**Input on Drive**:
- `swiss_law/research/stage_b_input/stage_b_input.parquet` (produced by `bundle_stage_b_input.py` locally)
- `swiss_law/research/anchor_funnel_val001_v7/all_targets.json`
- `swiss_law/research/anchor_funnel_val001_v7/gold_doc_sets.json`
- `swiss_law/research/concept_embedding_path/multi_aspect/val_aspects.parquet`
- `swiss_law/data/val.csv`

**Output on Drive**:
- `swiss_law/research/stage_b_grounded_llm/stage_b_survivors.parquet`  — per-(qid, did) decisions
- `swiss_law/research/stage_b_grounded_llm/stage_b_metrics.json`        — per-query precision/recall/F1

**Wall-clock**: ~30-60 min on G4 / RTX PRO 6000 Blackwell at TOP_K=500;
~2-4 hours at TOP_K=2000.

**Phase map**

| Phase | Purpose |
|---|---|
| 0 | Setup, paths, install |
| 1 | Load Stage B input + val queries + Stage 0 targets |
| 2 | Build per-candidate dossier prompts (1 prompt per (qid, did)) |
| 3 | LLM screen via vLLM batched generate |
| 4 | Parse strict-JSON output + apply hard gates (quote substring) |
| 5 | Per-query precision / recall / F1, with K-truncation by confidence |
| 6 | Save survivors + metrics |


## Phase 0 — Setup


In [ ]:
import os, sys, subprocess, json, time, gc, io, re
from pathlib import Path
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace")

IS_COLAB = "google.colab" in sys.modules
print(f"Colab: {IS_COLAB}")

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    subprocess.run([
        "pip", "install", "-q", "-U",
        "vllm>=0.9.1", "transformers>=4.51.0", "pandas==2.2.3",
        "pyarrow==16.1.0", "numpy==1.26.4", "tqdm",
    ], check=True)


## Phase 1 — Paths and data loading


In [ ]:
DRIVE_ROOT  = Path("/content/drive/MyDrive/swiss_law")
STAGE_B_IN  = DRIVE_ROOT / "research" / "stage_b_input" / "stage_b_input.parquet"
VAL_CSV     = DRIVE_ROOT / "data"     / "val.csv"
ALL_TARGETS = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "all_targets.json"
GOLD_SETS   = DRIVE_ROOT / "research" / "anchor_funnel_val001_v7" / "gold_doc_sets.json"
VAL_ASPECTS = DRIVE_ROOT / "research" / "concept_embedding_path"  / "multi_aspect" / "val_aspects.parquet"

OUT_DIR = DRIVE_ROOT / "research" / "stage_b_grounded_llm"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_SURVIVORS = OUT_DIR / "stage_b_survivors.parquet"
OUT_METRICS   = OUT_DIR / "stage_b_metrics.json"
OUT_RAW       = OUT_DIR / "stage_b_raw_llm.parquet"

# Tunables
TOP_K        = 2000            # raise recall ceiling from 0.39 (at TOP_K=500) to 0.65; binding rules below skip LLM on most rows, so compute stays bounded
LLM_MODEL    = "Qwen/Qwen3-32B-AWQ"
MAX_TOKENS   = 2048           # thinking trace ~1500 tokens + JSON ~150 + headroom; vLLM stops at EOS so unused budget costs nothing
MAX_MODEL_LEN= 8192            # prompt (~1500) + thinking (~1500) + output (~200) + headroom
BATCH_SIZE   = 256              # vLLM batches internally; this is the .generate() chunk size

print("Verifying inputs on Drive:")
for p in [STAGE_B_IN, VAL_CSV, ALL_TARGETS, GOLD_SETS, VAL_ASPECTS]:
    print(f"  {'OK' if p.exists() else 'MISSING'}  {p}")


In [ ]:
import pandas as pd

print("\nLoading val.csv ...")
val_df = pd.read_csv(VAL_CSV)
qid_to_query = {r.query_id: str(r.query) for r in val_df.itertuples()}
print(f"  {len(val_df)} queries")

print("Loading all_targets.json + gold_doc_sets.json ...")
targets   = json.load(open(ALL_TARGETS, encoding="utf-8"))
gold_sets = json.load(open(GOLD_SETS, encoding="utf-8"))

print("Loading val_aspects.parquet ...")
asp_df = pd.read_parquet(VAL_ASPECTS)
qid_to_aspects = {r.query_id: list(r.aspects) for r in asp_df.itertuples()}

print("Loading stage_b_input.parquet ...")
sb = pd.read_parquet(STAGE_B_IN)
print(f"  total rows: {len(sb):,}  cols: {len(sb.columns)}")

# Filter to top-K per query
sb = (
    sb.sort_values(["qid", "stage_a_rank"])
      .groupby("qid", group_keys=False)
      .head(TOP_K)
      .reset_index(drop=True)
)
print(f"  after top-{TOP_K} filter: {len(sb):,}")
print(f"  gold in retained rows: {sb['is_gold'].sum()}")

# Bind-LLM-to-dossier tiering.
# In v3 the LLM rejected 72% of gold that had strong dossier signals
# (article_match=Y, cc up to 9,975). Pure-dossier filters already match
# or beat the LLM. So: skip LLM on structurally certain rows, run it
# only on the ambiguous middle tier, discard rows with no signal at all.
sb["tier"] = "drop"
_auto = sb.article_match & ((sb.co_citation_count >= 30) | sb.code_in_target)
sb.loc[_auto, "tier"] = "auto"
_mid = (~_auto) & (
    sb.article_match
    | (sb.co_citation_count >= 5)
    | (sb.concept_cosine_score >= 0.55)
)
sb.loc[_mid, "tier"] = "llm"

print("\nTier breakdown:")
print(sb.groupby(["qid", "tier"]).size().unstack(fill_value=0).reindex(columns=["auto","llm","drop"], fill_value=0))
print("\nGold per tier (total across queries):")
print(sb.groupby(["tier","is_gold"]).size().unstack(fill_value=0).reindex(["auto","llm","drop"], fill_value=0))
print(f"\nLLM-tier candidates to score: {(sb.tier=='llm').sum():,}  (compute scales with this)")


## Phase 2 — Prompt construction


In [ ]:
PROMPT_TEMPLATE = """You are a senior Swiss lawyer evaluating whether to cite a specific legal source in the written legal opinion you would prepare to answer the question below. Reason carefully like a Swiss legal practitioner, using Swiss legal-system knowledge and the actual content of the candidate source.

LEGAL QUESTION FROM THE CLIENT:
{question}

THE QUESTION'S LEGAL ASPECTS (issues your opinion must address):
{aspects}

CANDIDATE SOURCE TO EVALUATE:
- Citation: {citation}
- Type: {family_label}
- Context: {family_extra}
- Paragraph role inside the source: {role}
- Substantive content of the source (in original language):
---BEGIN SOURCE TEXT---
{text}
---END SOURCE TEXT---

RETRIEVAL EVIDENCE FROM THE CORPUS (objective signals; weigh them, do not blindly follow):
- The query names this specific statute by article number: {article_match}
- This source is co-cited with {co_citation_count} other Swiss court paragraphs that match the query's pool
- Concept-cosine vs the query's legal aspects: {concept_cosine_score:.2f} (closest aspect: {best_aspect_id})
- The source's legal code/area matches the query's legal area: {code_in_target}

HOW A SWISS LAWYER REASONS ABOUT THIS:
1. Identify the legal subject matter and issues at stake in the question (e.g., pre-trial detention -> criminal procedure StPO; appeal deadlines -> BGG procedural rules; child support -> ZGB family law).
2. Enumerate the categories of sources a competent lawyer would cite in a memorandum on these issues:
   - foundational statutes (e.g., Art. 221 StPO for detention; Art. 285 ZGB for child support)
   - constitutional / treaty provisions when fundamental rights are at stake (BV, EMRK)
   - procedural rules that apply to any case of this type (Art. 100 BGG for any appeal; Art. 42 BGG for brief requirements)
   - leading BGE precedents on the specific issue
   - subsidiary/related provisions that competent counsel routinely cites alongside the main rule
3. Decide whether THIS candidate source falls into one of those legitimate citation categories given the question.

DECISION:
- KEEP = a competent Swiss lawyer writing a legal opinion on this question would cite this source (as foundational law, procedural rule, supporting BGE, OR a routinely co-cited adjacent provision). Be generous if structural evidence is strong: in Swiss legal practice, lawyers cite surrounding articles of the same statute (e.g., Art. 221 + 222 + 227 StPO for detention questions) even when only one is on point.
- REJECT only when the source is clearly outside the legal domain of the question, OR is purely procedural/administrative for a different case type, OR is a court paragraph from a manifestly unrelated chamber.

OUTPUT STRICT JSON (no prose before or after, begin with `{{`):
{{"keep": true|false, "confidence": <0.0-1.0>, "legal_role": "<foundational_statute|procedural_rule|constitutional_treaty|leading_precedent|adjacent_provision|background|off_topic>", "reasoning": "<one sentence in English explaining why a Swiss lawyer would or would not cite this in their opinion>"}}
"""

def format_aspects(aspects_list):
    if not aspects_list: return "(none extracted — treat as a single-aspect question)"
    lines = []
    for a in aspects_list:
        if isinstance(a, dict):
            label = a.get("label", "")
            w     = a.get("weight", 0)
            aid   = a.get("id", "")
            lines.append(f"  - {aid} (weight={float(w):.2f}): {label}")
    return "\n".join(lines) if lines else "(empty)"

def family_extra_str(row):
    if row.family == "court":
        return f"Swiss court paragraph. Court base: {row.court_base!s}. Chamber: {row.chamber!s}."
    else:
        return f"Swiss statute. Code: {row.law_code!s}. Law: {row.law_title!s}."

def family_label(row):
    return "Swiss court precedent paragraph" if row.family == "court" else "Swiss statutory provision"

def build_prompt(row, question, aspects_text):
    return PROMPT_TEMPLATE.format(
        question=question[:1500],
        aspects=aspects_text,
        citation=row.citation,
        family_label=family_label(row),
        family_extra=family_extra_str(row),
        role=row.role or "(unknown)",
        article_match=str(bool(row.article_match)).lower(),
        co_citation_count=int(row.co_citation_count),
        concept_cosine_score=float(row.concept_cosine_score),
        best_aspect_id=row.best_aspect_id or "?",
        code_in_target=str(bool(row.code_in_target)).lower(),
        text=(row.text or "")[:1200].replace('"', "'"),
    )

# Build prompts only for the LLM tier (middle-signal candidates).
sb_llm = sb[sb.tier == "llm"].reset_index(drop=True)
prompts = []
prompt_meta = []
qid_aspects_cache = {qid: format_aspects(qid_to_aspects.get(qid, [])) for qid in sb["qid"].unique()}
for r in sb_llm.itertuples():
    p = build_prompt(r, qid_to_query[r.qid], qid_aspects_cache[r.qid])
    prompts.append(p)
    prompt_meta.append((r.qid, r.did))

print(f"\nBuilt {len(prompts):,} LLM-tier prompts (lawyer-reasoning framing).")
print(f"  AUTO tier (no LLM call): {(sb.tier=='auto').sum():,}")
print(f"  DROP tier (no LLM call): {(sb.tier=='drop').sum():,}")
print(f"\nSample prompt for one LLM-tier candidate:\n{'-'*70}\n{prompts[0]}\n{'-'*70}")
print(f"\nPrompt length (chars): mean={int(sum(len(p) for p in prompts)/max(1,len(prompts))):,}  "
      f"max={max((len(p) for p in prompts), default=0):,}")


## Phase 3 — vLLM batched generation


In [ ]:
from vllm import LLM, SamplingParams

print(f"Loading {LLM_MODEL} on vLLM (thinking mode enabled) ...")
llm = LLM(
    model=LLM_MODEL,
    dtype="bfloat16",
    gpu_memory_utilization=0.60,
    max_model_len=MAX_MODEL_LEN,
    enforce_eager=False,
)

# Qwen3 THINKING sampling params from the Qwen3-32B HF model card:
#   "For thinking mode (enable_thinking=True, the default), we suggest using
#    Temperature=0.6, TopP=0.95, TopK=20, MinP=0."
# Reasoning trace can run 500-1500 tokens before the JSON; max_tokens must
# accommodate that plus the JSON payload (~150 tokens).
sp = SamplingParams(
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    max_tokens=MAX_TOKENS,
    seed=42,
    stop=["</json>"],
)

# Use llm.chat() with thinking ENABLED (the Qwen3 default).
# We omit chat_template_kwargs entirely so enable_thinking defaults to True.
messages_batch = [[{"role": "user", "content": p}] for p in prompts]

print(f"Generating for {len(prompts):,} prompts (thinking ON, ~3-5x slower than non-thinking) ...")
t0 = time.time()
outs = llm.chat(messages_batch, sampling_params=sp)
dt = time.time() - t0
print(f"Done in {dt/60:.1f} min  ({len(prompts)/max(1,dt):.1f} candidates/sec)")

raw_texts = [o.outputs[0].text if o.outputs else "" for o in outs]

print("--- Sample outputs (3 random) ---")
import random; random.seed(0)
for i in random.sample(range(len(raw_texts)), min(3, len(raw_texts))):
    print(f"[prompt {i}, len={len(raw_texts[i])}]")
    print(f"  {raw_texts[i][:600]!r}")
    print()

# Free LLM before downstream parsing
del llm
gc.collect()
import torch; torch.cuda.empty_cache()


## Phase 4 — Parse strict JSON + apply hard gates

Two gates:
1. **Quote substring**: `evidence_quote` MUST appear in the candidate's text (case-insensitive, whitespace-normalized). Drops hallucinated picks.
2. **Anti-optimism**: if `keep=true` but every dossier signal is 0 (no article match, no co-citation, no cosine ≥ 0.45, no code match), reject — pure LLM optimism with no evidence.


In [ ]:
def parse_lenient(s):
    """Lenient JSON parser. Strips think tokens if present, code fences, trailing commas."""
    s = s.strip()
    # Qwen3 thinking outputs may include <think>...</think> blocks; strip them.
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL).strip()
    if s.startswith("```"):
        s = s.split("\n", 1)[-1]
        if s.endswith("```"):
            s = s.rsplit("```", 1)[0]
    a, b = s.find("{"), s.rfind("}")
    if a == -1 or b == -1: return None
    for cand in [s[a:b+1], s[a:b+1].replace(",}", "}").replace(",]", "]")]:
        try: return json.loads(cand)
        except Exception: pass
    return None

CONF_FLOOR = 0.30   # only trust LLM keeps when its own confidence >= 0.30

# AUTO tier: forced keep, conf=1.0, no LLM call.
auto_rows = []
for r in sb[sb.tier == "auto"].itertuples():
    auto_rows.append({
        "qid": r.qid, "did": r.did, "is_gold": bool(r.is_gold),
        "stage_a_rank": int(r.stage_a_rank), "stage_a_score": float(r.stage_a_score),
        "citation": r.citation, "family": r.family,
        "tier": "auto", "keep_llm": True, "confidence": 1.0,
        "legal_role": "auto_dossier", "reasoning": "strong dossier signal — bypassed LLM",
        "keep_final": True, "parse_ok": True, "raw_llm": "",
    })

# LLM tier: parse JSON; final keep = LLM kept AND confidence >= floor.
llm_rows = []
for (qid, did), raw, src_row in zip(prompt_meta, raw_texts, sb_llm.itertuples()):
    parsed = parse_lenient(raw) or {}
    keep_llm = bool(parsed.get("keep", False))
    conf = float(parsed.get("confidence", 0.0) or 0.0)
    legal_role = str(parsed.get("legal_role", "") or "")
    reasoning = str(parsed.get("reasoning", "") or "")
    keep_final = keep_llm and (conf >= CONF_FLOOR)
    llm_rows.append({
        "qid": qid, "did": did, "is_gold": bool(src_row.is_gold),
        "stage_a_rank": int(src_row.stage_a_rank), "stage_a_score": float(src_row.stage_a_score),
        "citation": src_row.citation, "family": src_row.family,
        "tier": "llm", "keep_llm": keep_llm, "confidence": conf,
        "legal_role": legal_role[:60], "reasoning": reasoning[:300],
        "keep_final": keep_final, "parse_ok": bool(parsed), "raw_llm": raw[:1200],
    })

out_df = pd.DataFrame(auto_rows + llm_rows)
print(f"\nv5 output diagnostics:")
print(f"  AUTO kept (no LLM):        {(out_df.tier=='auto').sum()}")
print(f"  LLM tier total:            {(out_df.tier=='llm').sum()}")
if (out_df.tier=='llm').sum() > 0:
    sub = out_df[out_df.tier == "llm"]
    print(f"    parse_ok:                {sub.parse_ok.mean():.1%}")
    print(f"    keep_llm=true:           {sub.keep_llm.mean():.1%}")
    print(f"    confidence >= {CONF_FLOOR:.2f}:           {(sub.confidence >= CONF_FLOOR).mean():.1%}")
    print(f"    LLM-tier kept_final:     {sub.keep_final.mean():.1%}")
    print(f"    legal_role distribution:")
    print(sub.legal_role.value_counts().head(10).to_string())
print(f"  GRAND TOTAL kept_final:    {out_df.keep_final.sum()}")


## Phase 5 — Per-query precision / recall / F1

Reports two versions:
  - **No K cap**: keep all `keep_final == True` rows (lets LLM choose count).
  - **K-capped**: keep top-K_pred by confidence where K_pred = round(median of per-query gold counts).
Stage C will add proper adaptive K + aspect stratification.


In [ ]:
def f1(p, r):
    return 0.0 if (p + r) == 0 else 2 * p * r / (p + r)

print("\n=== Per-query results (no K cap — keep all keep_final=True) ===")
print(f"{'qid':<8}  {'gold':>5}  {'picks':>6}  {'correct':>8}  {'P':>6}  {'R':>6}  {'F1':>6}")
metrics_no_cap = {}
for qid in sorted(out_df.qid.unique()):
    sub  = out_df[out_df.qid == qid]
    gold = set(d for d, g in zip(sub.did, sub.is_gold) if g)
    # Gold count is total gold per query, not just gold in retained top-K
    total_gold = len(set(gold_sets.get(qid, [])))
    picks = set(sub[sub.keep_final].did)
    correct = len(picks & gold)
    p = correct / max(1, len(picks))
    r = correct / max(1, total_gold)
    metrics_no_cap[qid] = {"gold": total_gold, "picks": len(picks), "correct": correct,
                          "P": p, "R": r, "F1": f1(p, r)}
    print(f"  {qid:<8}  {total_gold:>5}  {len(picks):>6}  {correct:>8}  {p:>6.3f}  {r:>6.3f}  {f1(p,r):>6.3f}")
macro_p_nc = sum(m["P"]  for m in metrics_no_cap.values()) / len(metrics_no_cap)
macro_r_nc = sum(m["R"]  for m in metrics_no_cap.values()) / len(metrics_no_cap)
macro_f_nc = sum(m["F1"] for m in metrics_no_cap.values()) / len(metrics_no_cap)
print(f"\n  MACRO  P={macro_p_nc:.3f}  R={macro_r_nc:.3f}  F1={macro_f_nc:.3f}")

print("\n=== Per-query results (K-capped: top by confidence, K = predicted_gold_size) ===")
# Use median gold count as starter predicted size; Stage C will do better.
# For now, take top-K_q where K_q = total_gold (perfect-K oracle for upper bound)
print(f"{'qid':<8}  {'gold':>5}  {'picks':>6}  {'correct':>8}  {'P':>6}  {'R':>6}  {'F1':>6}")
metrics_kcap = {}
for qid in sorted(out_df.qid.unique()):
    sub = out_df[(out_df.qid == qid) & (out_df.keep_final)]
    sub = sub.sort_values("confidence", ascending=False)
    total_gold = len(set(gold_sets.get(qid, [])))
    # K = predicted_gold_size proxy: use total_gold for now (oracle-K). Stage C replaces this.
    K = total_gold
    picks = set(sub.head(K).did)
    gold_dids = set(gold_sets.get(qid, []))
    correct = len(picks & gold_dids)
    p = correct / max(1, len(picks))
    r = correct / max(1, total_gold)
    metrics_kcap[qid] = {"gold": total_gold, "K": K, "picks": len(picks),
                        "correct": correct, "P": p, "R": r, "F1": f1(p, r)}
    print(f"  {qid:<8}  {total_gold:>5}  {len(picks):>6}  {correct:>8}  {p:>6.3f}  {r:>6.3f}  {f1(p,r):>6.3f}")
macro_p_k = sum(m["P"]  for m in metrics_kcap.values()) / len(metrics_kcap)
macro_r_k = sum(m["R"]  for m in metrics_kcap.values()) / len(metrics_kcap)
macro_f_k = sum(m["F1"] for m in metrics_kcap.values()) / len(metrics_kcap)
print(f"\n  MACRO  P={macro_p_k:.3f}  R={macro_r_k:.3f}  F1={macro_f_k:.3f}")


## Phase 6 — Save outputs


In [ ]:
out_df.to_parquet(OUT_RAW, index=False)
print(f"Saved raw → {OUT_RAW}")

# Keep the slim survivors table separate
survivors = out_df[out_df.keep_final][["qid","did","citation","confidence","which_aspect",
                                       "evidence_quote","stage_a_rank","stage_a_score","is_gold"]]
survivors.to_parquet(OUT_SURVIVORS, index=False)
print(f"Saved survivors → {OUT_SURVIVORS}")

with open(OUT_METRICS, "w", encoding="utf-8") as f:
    json.dump({
        "config": {"TOP_K": TOP_K, "LLM_MODEL": LLM_MODEL, "MAX_TOKENS": MAX_TOKENS,
                   "BATCH_SIZE": BATCH_SIZE},
        "macro_no_cap":  {"P": macro_p_nc, "R": macro_r_nc, "F1": macro_f_nc},
        "macro_k_cap":   {"P": macro_p_k,  "R": macro_r_k,  "F1": macro_f_k},
        "per_query_no_cap": metrics_no_cap,
        "per_query_k_cap":  metrics_kcap,
        "diagnostics": {
            "parse_ok_rate":          float(out_df.parse_ok.mean()),
            "keep_llm_rate":          float(out_df.keep_llm.mean()),
            "quote_in_text_rate":     float(out_df.quote_in_text.mean()),
            "has_any_evidence_rate":  float(out_df.has_any_evidence.mean()),
            "keep_final_rate":        float(out_df.keep_final.mean()),
        },
    }, f, indent=2)
print(f"Saved metrics → {OUT_METRICS}")

print("\n=== DECISION GATE ===")
print(f"  Macro F1 (no K cap):     {macro_f_nc:.3f}")
print(f"  Macro F1 (K = K_gold):   {macro_f_k:.3f}  (upper bound — Stage C will pick K adaptively)")
print()
if macro_f_k >= 0.60:
    print("  ✅ Stage B passed (F1 >= 0.60). Proceed to Stage C (adaptive K + aspect stratification).")
elif macro_f_k >= 0.50:
    print("  ⚠ Stage B partial pass (0.50-0.60). Diagnose:")
    print("    - If keep_llm_rate > 0.5 → LLM over-permissive; tighten prompt rules.")
    print("    - If quote_in_text rate < 0.7 → LLM hallucinating quotes; tighten quote rule.")
    print("    - If parse_ok < 0.95 → JSON output format unreliable; consider guided_json.")
else:
    print("  ❌ Stage B failed (F1 < 0.50). Likely causes:")
    print("    - Stage A's top-K had too little gold (R@K too low).")
    print("    - Dossier signals not surfaced clearly enough in prompt.")
    print("    - Consider widening TOP_K from 500 to 1000+.")
